# 📝 Speech Transcription from Interview Audio  
*(This notebook can be executed end-to-end by selecting “Run all”)*

### Step 3 in the Agent-Based Digital Twin Demo

This notebook is the **third step** in the hands-on demonstration accompanying the paper  
**“An Agent-Based Digital Twin for Transparent AI in Student Admissions.”**

In the full system, speech transcription is performed by a dedicated AI agent that operates **in parallel** with video-based and document-based agents.  
For demonstration and reproducibility in Google Colab, we execute these agents **sequentially**, using separate notebooks that together simulate the full workflow.

---

## Where This Notebook Fits in the Pipeline

- **Step 1** – Mock interview video recording  
  👉 https://colab.research.google.com/drive/1mqcCrypaK3u_kvs7nSqX4XTvBgq4-shK

- **Step 2** – Audio extraction from interview videos  
  👉 https://colab.research.google.com/drive/1jTYrU0brHLeMTI65ZzKUDjL8iiCGddPp

- **Step 3 (this notebook)** – Speech transcription from extracted audio

Later notebooks use these transcriptions for:
- answer quality analysis,
- fluency and consistency checks,
- alignment with video-based behavioral signals.

---

## What This Notebook Does

This notebook applies an automatic speech recognition (ASR) agent to the extracted interview audio files.

Specifically, it:
- scans application folders for `.mp3` audio files,
- transcribes each audio file using a local Whisper model,
- saves plain-text transcripts alongside the original data,
- preserves folder structure to maintain traceability.

Each transcript becomes an inspectable artifact used by downstream agents and by human reviewers.


In [ ]:
import os
from google.colab import drive

if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
else:
    print("✅ Google Drive is already mounted")

✅ Google Drive is already mounted


In [ ]:
import os

drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"
os.makedirs(drive_dir, exist_ok=True)

subdirs = [d for d in os.listdir(drive_dir) if os.path.isdir(os.path.join(drive_dir, d))]
print(subdirs)

['0_0', '18190_27580', '40529_33700', '60811_46042']


In [ ]:
from pathlib import Path

drive_dir = "/content/drive/MyDrive/colab-output/dream_applicant_application/"

# Dictionary to store files by subdirectory
files_by_subdir = {}

# Iterate through direct subdirectories
for subdir in Path(drive_dir).iterdir():
    if subdir.is_dir():
        # Check if 'video_interviews' exists in this subdirectory
        documents_video_path = subdir / "video_interviews_audio_files"

        if documents_video_path.exists() and documents_video_path.is_dir():
            # Store both direct files and files in subdirectories
            video_info = {
                'direct_files': [],
                'subdirs': {}
            }

            # Iterate through contents of video_interviews
            for item in documents_video_path.iterdir():
                if item.is_file():
                    # Direct video file
                    video_info['direct_files'].append(item)
                elif item.is_dir():
                    # Subdirectory with video files
                    subdir_files = [f for f in item.iterdir() if f.is_file()]
                    if subdir_files:
                        video_info['subdirs'][item.name] = subdir_files

            files_by_subdir[subdir.name] = video_info

            total_files = len(video_info['direct_files']) + sum(len(files) for files in video_info['subdirs'].values())
            print(f"Found {total_files} files in {subdir.name}/video_interviews/")

# Now you can use the files later
for subdir_name, video_info in files_by_subdir.items():
    print(f"\n{subdir_name}:")

    # Print direct files
    if video_info['direct_files']:
        print("  Direct files:")
        for file in video_info['direct_files']:
            print(f"    - {file.name}")

    # Print files in subdirectories
    if video_info['subdirs']:
        print("  Files in subdirectories:")
        for subdir_name, files in video_info['subdirs'].items():
            print(f"    {subdir_name}/")
            for file in files:
                print(f"      - {file.name}")

Found 7 files in 60811_46042/video_interviews/

60811_46042:
  Direct files:
    - 51375_What_new_hobbies_or_activities_would_you_like_to_t.mp3
    - 51376_Which_qualities_of_VILNIUS_TECH_stood_out_to_you_a.mp3
    - 51377_How_would_your_friends_characterize_you_.mp3
    - 51378_How_do_you_deal_with_unexpected_changes_or_disrupt.mp3
    - 51379_What_does_innovation_mean_to_you_personally__how_d.mp3
    - 51380_What_methods_do_you_use_to_make_sure_you_stay_on_t.mp3
    - 51381_What_made_you_choose_this_specific_programme_at_VI.mp3


In [ ]:
pip -q install openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 18.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import os
import whisper
from pathlib import Path

# Load Whisper model
print("Loading Whisper model...")
model = whisper.load_model("large")  # change to "small", "base", medium "large" if needed
print("Model loaded ✓\n")

# Transcribe audio files
for subdir_name, video_info in files_by_subdir.items():
    print(f"\nProcessing transcriptions for {subdir_name}...")

    # Create transcription directory path
    transcription_base_dir = Path(drive_dir) / subdir_name / "video_interviews_audio_transcriptions_text"

    has_transcription_files = False

    # Process direct audio files
    if video_info['direct_files']:
        for video_file in video_info['direct_files']:
            # Prepare audio and transcription file paths
            audio_filename = video_file.stem + ".mp3"
            audio_path = Path(drive_dir) / subdir_name / "video_interviews_audio_files" / audio_filename

            text_filename = video_file.stem + ".txt"
            text_path = transcription_base_dir / text_filename

            # Check if transcription already exists
            if text_path.exists():
                print(f"  ⏭️  Skipping {audio_filename} (transcription already exists)")
                has_transcription_files = True
                continue

            # Check if audio file exists
            if not audio_path.exists():
                print(f"  ⚠️  Audio file not found: {audio_filename}")
                continue

            # Create transcription directory if needed
            if not has_transcription_files:
                os.makedirs(transcription_base_dir, exist_ok=True)
                has_transcription_files = True

            # Transcribe audio
            try:
                print(f"  📝 Transcribing {audio_filename}")
                #result = model.transcribe(str(audio_path))
                result = model.transcribe(str(audio_path), language="en", condition_on_previous_text=False)

                with open(text_path, "w", encoding="utf-8") as f:
                    f.write(result["text"])

                print(f"     ✓ Saved transcription")
            except Exception as e:
                print(f"  ❌ Error transcribing {audio_filename}: {e}")

    # Process files in subdirectories
    if video_info['subdirs']:
        for subdir_folder_name, files in video_info['subdirs'].items():
            # Create corresponding subdirectory in transcription folder
            transcription_subdir = transcription_base_dir / subdir_folder_name
            audio_subdir = Path(drive_dir) / subdir_name / "video_interviews_audio_files" / subdir_folder_name

            for video_file in files:
                # Prepare audio and transcription file paths
                audio_filename = video_file.stem + ".mp3"
                audio_path = audio_subdir / audio_filename

                text_filename = video_file.stem + ".txt"
                text_path = transcription_subdir / text_filename

                # Check if transcription already exists
                if text_path.exists():
                    print(f"  ⏭️  Skipping {subdir_folder_name}/{audio_filename} (transcription already exists)")
                    has_transcription_files = True
                    continue

                # Check if audio file exists
                if not audio_path.exists():
                    print(f"  ⚠️  Audio file not found: {subdir_folder_name}/{audio_filename}")
                    continue

                # Create transcription subdirectory if needed
                if not has_transcription_files:
                    os.makedirs(transcription_base_dir, exist_ok=True)
                    has_transcription_files = True
                os.makedirs(transcription_subdir, exist_ok=True)

                # Transcribe audio
                try:
                    print(f"  📝 Transcribing {subdir_folder_name}/{audio_filename}")
                    #result = model.transcribe(str(audio_path))
                    result = model.transcribe(str(audio_path), language="en", condition_on_previous_text=False)

                    with open(text_path, "w", encoding="utf-8") as f:
                        f.write(result["text"])

                    print(f"     ✓ Saved transcription")
                except Exception as e:
                    print(f"  ❌ Error transcribing {subdir_folder_name}/{audio_filename}: {e}")

print("\n✅ All transcriptions completed!")

Loading Whisper model...


100%|█████████████████████████████████████| 2.88G/2.88G [00:34<00:00, 90.5MiB/s]


Model loaded ✓


Processing transcriptions for 60811_46042...
  📝 Transcribing 51375_What_new_hobbies_or_activities_would_you_like_to_t.mp3
     ✓ Saved transcription
  📝 Transcribing 51376_Which_qualities_of_VILNIUS_TECH_stood_out_to_you_a.mp3
     ✓ Saved transcription
  📝 Transcribing 51377_How_would_your_friends_characterize_you_.mp3
     ✓ Saved transcription
  📝 Transcribing 51378_How_do_you_deal_with_unexpected_changes_or_disrupt.mp3
     ✓ Saved transcription
  📝 Transcribing 51379_What_does_innovation_mean_to_you_personally__how_d.mp3
     ✓ Saved transcription
  📝 Transcribing 51380_What_methods_do_you_use_to_make_sure_you_stay_on_t.mp3
     ✓ Saved transcription
  📝 Transcribing 51381_What_made_you_choose_this_specific_programme_at_VI.mp3
     ✓ Saved transcription

✅ All transcriptions completed!


---

▶️ Next Steps in the Digital Twin Demo

In the full admissions process digital twin, speech transcription is only one component of interview analysis.
Additional agents operate on the same interview data to examine:

temporal consistency between speech and lip motion,

indicators of reading or scripted responses,

semantic relevance and structure of answers.

In a production deployment, these agents execute in parallel as part of a coordinated multi-agent workflow.
For the purposes of this Google Colab demonstration, the workflow is flattened into a sequence of notebooks, each representing one agent or processing stage.

Step 4: Transcript-Based Interview Evaluation

The next notebook applies a prompt-driven evaluation to each interview transcript, focusing on:

content relevance and clarity,

naturalness versus scripted delivery,

linguistic markers associated with AI-generated text.

👉 Proceed to the next notebook:
https://colab.research.google.com/drive/1VZXmuQiw5mCzvcPNE4aqbMLn5OZ1R7tY?usp=sharing

Subsequent notebooks further fuse textual, audio, and visual evidence into a unified applicant profile, supporting transparent inspection of the end-to-end AI-assisted admissions process.

Note:
A public GitHub repository containing all notebooks, prompts, and orchestration code will be released after the anonymous review process is completed.




